# ML-09 — Validation Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

## 1. Two paper findings + methodology audit

**Finding 1: "Declining pages with demand"

Label: `trend_direction == "down"` + `impressions >= 100`

Validation: 80.9% of declining pages clear impression bar (vs 73.4% base). Holds up. But no future data—can't validate if recovery happened after review.

---

**Finding 2: "Position/impressions structure CTR opportunity"

Label: Rule-based proxy (position ≤20 + impressions ≥500 + CTR <0.5%), not observed outcome.

Validation: **Circular.** RF top 2 features are position (0.351) and impressions (0.364)—the label definition itself. Precision@50=0.98 proves model learned rule, not discovered signal.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
q = df[(df.impressions_90d > 0) & (df.content_age_days >= 90) & (df.avg_position > 0)].copy()

label = ((q.impressions_90d >= 500) & (q.avg_position <= 20) & (q.ctr < 0.5)).astype(int)

print("CIRCULARITY CHECK:")
print(f"Label rate: {label.mean():.3f}")
print()
print("AUC vs label (features literally IN label):")
print(f"  avg_position: {roc_auc_score(label, -q.avg_position):.3f}")
print(f"  impressions_90d: {roc_auc_score(label, q.impressions_90d):.3f}")
print()
print("AUC vs label (features NOT in label):")
print(f"  word_count: {roc_auc_score(label, q.word_count.fillna(0)):.3f}")
print()
print("→ Model's top 2 features ARE the label. Not discovery—overfit.")


## 2. Honest split (before/after)

**Before (random split):** Precision@50 = 0.98 (overfitted)

**After (client-grouped):** Clients don't leak between train/test; honest score.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

num_f = ["avg_position", "impressions_90d", "word_count", "content_age_days", "days_since_last_update", "scroll_rate", "engagement_rate"]
cat_f = ["content_type", "main_intent", "competition_level", "freshness_tier"]
q["has_wc"] = q.word_count.notna().astype(int)

X = q[num_f + ["has_wc"] + cat_f].copy()
y = label.values

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_f + ["has_wc"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), cat_f)
])

def prec_k(yt, yp, k):
    top = np.argsort(-yp)[:k]
    return yt[top].mean()

# Random
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
rf1 = Pipeline([("p", prep), ("c", RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42))])
rf1.fit(Xtr, ytr)
p1 = rf1.predict_proba(Xte)[:, 1]

# Grouped
gs = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
ti, tei = next(gs.split(X, groups=q.client_id))
rf2 = Pipeline([("p", prep), ("c", RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42))])
rf2.fit(X.iloc[ti], y[ti])
p2 = rf2.predict_proba(X.iloc[tei])[:, 1]

print("SPLIT AUDIT:")
print(f"Random: @20={prec_k(yte, p1, 20):.3f}, @50={prec_k(yte, p1, 50):.3f}")
print(f"Grouped: @20={prec_k(y[tei], p2, 20):.3f}, @50={prec_k(y[tei], p2, 50):.3f}")
print(f"  (Train: {len(ti)} rows, {q.iloc[ti].client_id.nunique()} clients | Test: {len(tei)} rows, {q.iloc[tei].client_id.nunique()} clients)")


## 3. Leakage audit

No temporal leakage (no future-window fields). But structural leakage: position & impressions are in label, causing model to memorize rule.

In [ ]:
label_parts = {"avg_position", "impressions_90d", "ctr"}
features = set(["avg_position", "impressions_90d", "word_count", "content_age_days", "days_since_last_update", "scroll_rate", "engagement_rate"])

print("Leakage check:")
print(f"  Label components in features: {label_parts & features}")
print(f"  → Structural leakage (causes overconfidence)")
print(f"  → Honest model: drop position/impressions, use only secondary signals")


## 4. Claim rewrite

**Bold:** "Position and impressions strongly structure CTR opportunity."

**Safe:** "Pages in positions 1–20 with ≥500 impressions and <0.5% CTR cluster together (33.9% of data). The pattern is well-defined; whether it reflects a fixable metadata issue requires future-outcome validation."

Key: removed causal language, added "in this snapshot", opened question about fix.

In [ ]:
claims = pd.DataFrame({
    "Bold": [
        "Engagement metrics predict low CTR.",
        "The model identifies pages to rewrite."
    ],
    "Safe": [
        "Engagement metrics correlate with low-CTR pattern (r<0.15, modest signal).",
        "The model ranks by pattern fit. Humans verify & decide on rewrite."
    ]
})
print(claims.to_string())


## Self-check

- [x] All sections filled
- [ ] Runs top to bottom
- [x] No client names/URLs
- [x] Safe language
- [ ] Committed to repo